In [ ]:
# Dependencies (unsloth, unsloth_zoo, sentencepiece, protobuf, hf_transfer,
# xformers, trl, torchvision, transformers, bitsandbytes, accelerate, peft,
# datasets) are managed by UV via `pyproject.toml`.
#
# From the repository root, install them with:
#
#     uv sync --extra unsloth
#
# Then launch Jupyter from the same environment (e.g. `uv run jupyter lab`)
# so this notebook can import unsloth without any in-notebook `pip install`.

In [ ]:
from unsloth import FastModel
from transformers import AutoModelForSequenceClassification
import torch

# Compat shim: torchvision cu128 wheels are built without the video backend,
# so `torchvision.io.VideoReader` is missing. `datasets==4.3.0` (pinned by
# unsloth) imports it unconditionally in its Torch formatter, which breaks
# `trainer.train()`. Stub a dummy class so the isinstance(...) branch is a
# harmless no-op (we do not use video in this notebook).
import torchvision.io
if not hasattr(torchvision.io, "VideoReader"):
    class VideoReader:  # noqa: N801 - mirror torchvision's public name
        pass
    torchvision.io.VideoReader = VideoReader

%env UNSLOTH_DISABLE_FAST_GENERATION = 1
max_seq_length = 256
dtype = None
load_in_4bit = False

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HF_NEW")
assert hf_token, "Set HF_TOKEN (or HF_NEW) in your .env file at the repo root"

# ---- Weights & Biases: comprehensive experiment tracking -------------------
# Follows the official HF + W&B integration guide
# (https://docs.wandb.ai/models/integrations/huggingface_transformers).
import wandb
from datetime import datetime

wandb_api_key = os.environ.get("WANDB_API_KEY")
if wandb_api_key:
    wandb.login(key=wandb_api_key)

# Trainer-level integration knobs (read by transformers.integrations.WandbCallback).
# Must be set BEFORE the Trainer is instantiated.
os.environ.setdefault("WANDB_WATCH", "all")      # gradients + params histograms
os.environ.setdefault("WANDB_LOG_MODEL", "false") # we upload the final model manually
os.environ.setdefault("WANDB_SILENT", "false")

run_name = os.environ.get(
    "WANDB_NAME",
    f"embedding-classifier-e5-large-{datetime.utcnow():%Y%m%d-%H%M%S}",
)

wandb_run = wandb.init(
    project=os.environ.get("WANDB_PROJECT", "lid-bench"),
    entity=os.environ.get("WANDB_ENTITY"),
    name=run_name,
    group="embedding-classifier",
    job_type="train",
    tags=["embedding", "xlm-roberta-large", "multilingual-e5-large", "lid-67"],
    config={
        "model_name": "intfloat/multilingual-e5-large",
        "max_seq_length": max_seq_length,
        "load_in_4bit": load_in_4bit,
        "dtype": str(dtype) if dtype is not None else "auto",
        "dataset": "1024m/LID",
        "dataset_file": "Data_Hackathon/LID-1000.parquet",
    },
    save_code=True,
    reinit=True,
)

# Define how each metric should be summarized in the run summary panel.
# Trainer pushes eval metrics under eval_<dataset>/<metric_name>.
for prefix in ("eval_val",):
    wandb.define_metric(f"{prefix}/macro_f1", summary="max")
    wandb.define_metric(f"{prefix}/micro_f1", summary="max")
    wandb.define_metric(f"{prefix}/weighted_f1", summary="max")
    wandb.define_metric(f"{prefix}/accuracy", summary="max")
    wandb.define_metric(f"{prefix}/macro_precision", summary="max")
    wandb.define_metric(f"{prefix}/macro_recall", summary="max")
    wandb.define_metric(f"{prefix}/loss", summary="min")
wandb.define_metric("train/loss", summary="min")
wandb.define_metric("train/learning_rate", summary="last")

print(f"W&B run: {wandb_run.url}")


In [ ]:
from datasets import load_dataset
LOAD_SPECIFIC_FILE = True
dataset_name = "1024m/LID"
if LOAD_SPECIFIC_FILE:
    file_path = "Data_Hackathon/LID-1000.parquet"
    dataset = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=hf_token)["train"]
else:
    dataset = load_dataset(dataset_name, token=hf_token)["train"]
print(f"dataset  : {dataset_name}")
print(f"samples  : {len(dataset)}")
print(f"columns  : {dataset.column_names}")
print(f"size     : {dataset.dataset_size / 1024**2:.3f} MB")

In [ ]:
NUM_LABELS = len(dataset.unique("ISO-693-3"))
print(NUM_LABELS)
print(dataset.unique("ISO-693-3"))

In [ ]:
labels = sorted(dataset.unique("ISO-693-3"))
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

In [ ]:
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"
import torch.nn as nn
model, tokenizer = FastModel.from_pretrained(
    model_name = "intfloat/multilingual-e5-large",
    auto_model = AutoModelForSequenceClassification,
    max_seq_length = max_seq_length,
    dtype = dtype,
    full_finetuning = True,
    load_in_4bit = load_in_4bit,
)
if hasattr(model.classifier, "out_proj"):
    model.classifier.out_proj = nn.Linear(model.classifier.out_proj.in_features, NUM_LABELS, bias=True).to(model.device).to(model.dtype)
else:
    model.classifier = nn.Linear(model.classifier.in_features, NUM_LABELS, bias=True).to(model.device).to(model.dtype)
model.config.num_labels = NUM_LABELS
model.config.id2label = id2label
model.config.label2id = label2id

# Unsloth patches XLM-Roberta to use the `flex_attention` backend, which
# raises `ValueError: flex_attention does not support dropout` during
# training. `intfloat/multilingual-e5-large` ships with
# `attention_probs_dropout_prob=0.1`, so zero it out on the config *and*
# on every already-instantiated self-attention layer before trainer.train().
model.config.attention_probs_dropout_prob = 0.0
for _m in model.modules():
    if _m.__class__.__name__.endswith("SelfAttention") and hasattr(_m, "dropout") and hasattr(_m.dropout, "p"):
        _m.dropout.p = 0.0

In [ ]:
model = FastModel.get_peft_model(model, r = 8, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
                                 lora_alpha = 16, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth",
                                 random_state = 1024, use_rslora = False, loftq_config = None, task_type = "SEQ_CLS",)

In [ ]:
from datasets import ClassLabel
if isinstance(dataset, dict):
    dataset = dataset["train"]
dataset = dataset.cast_column("ISO-693-3", ClassLabel(names=sorted(dataset.unique("ISO-693-3"))))
dataset = dataset.train_test_split(test_size=0.1, stratify_by_column="ISO-693-3")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=max_seq_length)
train_dataset = dataset['train'].map(tokenize_function, batched=True, num_proc=16)
val_dataset = dataset["test"].map(tokenize_function, batched=True, num_proc=16)
print(len(train_dataset))
print(len(val_dataset))

In [ ]:
train_dataset = train_dataset.rename_column("ISO-693-3", "label")
val_dataset = val_dataset.rename_column("ISO-693-3", "label")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
labels = train_dataset["label"]
class_weights = compute_class_weight("balanced", classes = np.unique(labels), y = labels)

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
train_dataset = train_dataset.remove_columns(["text", "lang", "source"])
val_dataset = val_dataset.remove_columns(["text", "lang", "source"])
train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [ ]:
import json
def process_benchmark(file_path):
    ds = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=hf_token)["train"]
    lang_col = next(c for c in ds.column_names if c.lower() == "iso-693-3")
    text_col = next(c for c in ds.column_names if c.lower() == "text")
    ds = ds.filter(lambda x: x[lang_col] in label2id)
    ds = ds.map(lambda x: {"labels": label2id[x[lang_col]]})
    if text_col != "text":
        ds = ds.rename_column(text_col, "text")
    ds = ds.map(tokenize_function, batched=True, num_proc=16)
    keep = [c for c in ["input_ids", "attention_mask", "token_type_ids", "labels"] if c in ds.column_names]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    ds.set_format("torch")
    return ds
benchmark_files = {
    "CommonLID": "Data_Benchmarks_Filtered/filtered_benchmark_CommonLID.parquet",
    "FLORES":    "Data_Benchmarks_Filtered/filtered_benchmark_FLORES.parquet",
    "SmolSent":  "Data_Benchmarks_Filtered/filtered_benchmark_SmolSent.parquet",
    "UDHRLID":   "Data_Benchmarks_Filtered/filtered_benchmark_UDHRLID.parquet",
}
benchmark_datasets = {name: process_benchmark(path) for name, path in benchmark_files.items()}
for name, ds in benchmark_datasets.items():
    print(f"{name}: {len(ds)} samples")

In [ ]:
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
)
import numpy as np
import json
import wandb

eval_context = {"dataset_name": "val", "step": 0}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    metrics = {
        "macro_f1":        f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1":        f1_score(labels, preds, average="micro", zero_division=0),
        "weighted_f1":     f1_score(labels, preds, average="weighted", zero_division=0),
        "accuracy":        accuracy_score(labels, preds),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall":    recall_score(labels, preds, average="macro", zero_division=0),
    }

    # Per-language accuracy (dict -> JSON file + wandb.Table).
    per_label = {}
    per_lang_rows = []
    for lid in np.unique(labels):
        mask = labels == lid
        lang = id2label[int(lid)]
        lang_acc = float(accuracy_score(labels[mask], preds[mask]))
        per_label[lang] = f"{lang_acc:.3f}"
        per_lang_rows.append([lang, lang_acc, int(mask.sum())])

    name = eval_context["dataset_name"]
    step = eval_context["step"]
    with open(f"{name}-{step}-SCORES.json", "w") as f:
        json.dump(per_label, f, indent=2)

    if wandb.run is not None:
        per_lang_table = wandb.Table(
            columns=["language", "accuracy", "n_samples"],
            data=per_lang_rows,
        )
        payload = {
            f"charts/{name}/per_lang_accuracy_table": per_lang_table,
            f"charts/{name}/per_lang_accuracy_bar": wandb.plot.bar(
                per_lang_table, "language", "accuracy",
                title=f"Per-language accuracy ({name}, step {step})",
            ),
        }
        # Confusion matrix is informative but expensive with 67 classes; only
        # log on the primary val set to keep logs lean.
        if name == "val":
            class_names = [id2label[i] for i in range(NUM_LABELS)]
            payload[f"charts/{name}/confusion_matrix"] = wandb.plot.confusion_matrix(
                y_true=labels.tolist(),
                preds=preds.tolist(),
                class_names=class_names,
                title=f"Confusion matrix ({name}, step {step})",
            )
        wandb.log(payload, step=step)

    return metrics


In [ ]:
from transformers import TrainingArguments, Trainer
from unsloth import is_bfloat16_supported
import torch

class LIDTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        eval_context["step"] = self.state.global_step
        if eval_dataset is None and isinstance(self.eval_dataset, dict):
            all_metrics = {}
            for name, ds in self.eval_dataset.items():
                eval_context["dataset_name"] = name
                m = super().evaluate(
                    eval_dataset=ds,
                    ignore_keys=ignore_keys,
                    metric_key_prefix=f"eval_{name}",
                )
                all_metrics.update(m)
            return all_metrics
        eval_context["dataset_name"] = metric_key_prefix
        return super().evaluate(
            eval_dataset=eval_dataset,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
        )

eval_datasets = {"val": val_dataset, **benchmark_datasets}

trainer = LIDTrainer(
    model=model,
    processing_class=tokenizer,
    eval_dataset=eval_datasets,
    train_dataset=train_dataset,
    args=TrainingArguments(
        # Memory-tuned for ~24 GB GPU (original A100-80GB config was
        # per_device_train_batch_size=1440 / grad_accum=1). We preserve the
        # effective batch (32 * 45 = 1440) via gradient accumulation and
        # enable gradient checkpointing to shrink activation memory ~10x.
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=45,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        eval_strategy="steps",
        eval_steps=0.1,
        lr_scheduler_type="linear",
        seed=1024,
        output_dir="outputs",
        # Full W&B logging (docs: "the most important step"):
        report_to="wandb",
        run_name=wandb_run.name if wandb_run is not None else None,
        logging_dir="outputs/runs",
        log_level="info",
    ),
    compute_metrics=compute_metrics,
)

# Runtime-derived config not available at wandb.init() time.
if wandb.run is not None:
    wandb.config.update({
        "effective_batch_size": trainer.args.per_device_train_batch_size
                                 * trainer.args.gradient_accumulation_steps,
        "num_labels": NUM_LABELS,
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        **{f"benchmark_samples/{k}": len(v) for k, v in benchmark_datasets.items()},
        "class_weights_mean": float(np.mean(class_weights)),
        "class_weights_min": float(np.min(class_weights)),
        "class_weights_max": float(np.max(class_weights)),
    }, allow_val_change=True)


In [ ]:
trainer_stats = trainer.train()

# Final summary: pin the most important numbers to the run summary panel
# so they show up in the W&B Runs table.
if wandb.run is not None:
    summary = {
        "final/train_runtime_sec":         float(trainer_stats.metrics.get("train_runtime", 0.0)),
        "final/train_samples_per_second":  float(trainer_stats.metrics.get("train_samples_per_second", 0.0)),
        "final/train_steps_per_second":    float(trainer_stats.metrics.get("train_steps_per_second", 0.0)),
        "final/train_loss":                float(trainer_stats.training_loss),
        "final/total_flos":                float(trainer_stats.metrics.get("total_flos", 0.0)),
        "final/epoch":                     float(trainer_stats.metrics.get("epoch", 0.0)),
    }
    if torch.cuda.is_available():
        summary["final/gpu_mem_peak_mb"]     = torch.cuda.max_memory_allocated() / 1e6
        summary["final/gpu_mem_reserved_mb"] = torch.cuda.max_memory_reserved() / 1e6
    wandb.run.summary.update(summary)


In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)
test_text = dataset["test"][0]["text"]
true_label = id2label[int(val_dataset[0]["labels"])]
result = classifier(test_text, truncation=True, max_length=256)
print(f"text       : {test_text[:100]}")
print(f"true label : {true_label}")
print(f"predicted  : {result[0]['label']} ({result[0]['score']:.4f})")

if wandb.run is not None:
    sample_table = wandb.Table(
        columns=["text", "true_label", "predicted_label", "confidence"],
        data=[[test_text[:500], true_label, result[0]["label"], float(result[0]["score"])]],
    )
    wandb.log({"inference/sanity_check": sample_table})


In [ ]:
save_dir = "baseline_lora_v2"
model.save_pretrained(save_dir)  # Local saving
tokenizer.save_pretrained(save_dir)

if wandb.run is not None:
    artifact = wandb.Artifact(
        name=f"model-{wandb.run.name}",
        type="model",
        description="Fine-tuned multilingual-e5-large for 67-language LID",
        metadata={
            "base_model":      "intfloat/multilingual-e5-large",
            "num_labels":      NUM_LABELS,
            "max_seq_length":  max_seq_length,
            "effective_batch": 32 * 45,
            "final_train_loss": float(trainer_stats.training_loss),
        },
    )
    artifact.add_dir(save_dir)
    wandb.run.log_artifact(artifact, aliases=["latest", "final"])


In [ ]:
model.push_to_hub("cataluna84/LIDL1Bv3", token=hf_token)       # Online saving
tokenizer.push_to_hub("cataluna84/LIDL1Bv3", token=hf_token)   # Online saving

if wandb.run is not None:
    wandb.run.summary["hf_model_url"] = "https://huggingface.co/cataluna84/LIDL1Bv3"
    wandb.finish()   # Required in notebooks to close the run cleanly.
